# Preprocessing and Exploratory Data Analysis
In this notebook, we will load the data, show some graphs before processing, process it, show some graphs after processing, and save the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add the src folder to path so we can import our custom files
sys.path.append(os.path.abspath('..'))

from src.preprocessing.clean_data import load_raw_data, impute_missing_values
from src.preprocessing.encoders import one_hot_encode_categorical, min_max_scale_numeric
from src.preprocessing.imbalance import random_undersample

# Create directories for saving graphs
os.makedirs('../graphs/raw', exist_ok=True)
os.makedirs('../graphs/preprocessed', exist_ok=True)

print("Libraries loaded and graph folders created!")

## 1. Load the Data
We use pd.read_excel because the file is an Excel file.

In [ ]:
filepath = "../data/raw/E Commerce Dataset.xlsx"
df = load_raw_data(filepath)

print("Data shape:", df.shape)
df.head()

## 2. Graphs Before Processing
Let's see how the data looks before we do anything to it.

In [ ]:
# Graph 1: Churn Imbalance
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Churn")
plt.title("Before Processing: Churn Imbalance")
plt.savefig("../graphs/raw/01_churn_imbalance.png", bbox_inches='tight')
plt.show()

**Insight:** Most of the customers in the dataset didn't churn (class 0), so it's super imbalanced. If we don't fix this, our model is basically just gonna guess 'not churned' every time. We need to undersample to make it fair.

In [ ]:
# Graph 2: Distribution of Tenure
plt.figure(figsize=(6,4))
sns.histplot(df["Tenure"], bins=20, kde=True)
plt.title("Before Processing: Tenure Distribution")
plt.savefig("../graphs/raw/02_tenure_distribution.png", bbox_inches='tight')
plt.show()

**Insight:** The tenure graph is right-skewed. Looks like most people haven't been using the app for very long, but there are a few really loyal old customers hanging around.

In [ ]:
# Graph 3: Cashback Amount vs Churn
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="Churn", y="CashbackAmount")
plt.title("Before Processing: Cashback Amount vs Churn")
plt.savefig("../graphs/raw/03_cashback_vs_churn.png", bbox_inches='tight')
plt.show()

**Insight:** The cashback boxplot shows that people who churned had slightly different cashback amounts compared to those who stayed, but it's pretty mixed. Also, look at all those dots past the whiskers - we definitely have some crazy outliers here.

## 2.1 Outlier Detection
Before processing, we must explicitly check for outliers in continuous numerical variables like `CashbackAmount` and `Tenure` using boxplots and IQR calculations.

In [ ]:
# Checking outliers using Boxplots and calculating IQR
continuous_cols = ['CashbackAmount', 'Tenure']
plt.figure(figsize=(10, 4))
for i, col in enumerate(continuous_cols, 1):
    plt.subplot(1, 2, i)
    sns.boxplot(y=df[col])
    plt.title(f"Outlier Check: {col}")
plt.tight_layout()
plt.savefig("../graphs/raw/04_outliers_check.png", bbox_inches='tight')
plt.show()

# IQR Calculation to show where outliers lie
for col in continuous_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        print(f"{col} - Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}, Outliers Count: {len(outliers)}")


**Insight:** Doing the math with IQR and boxplots shows we really do have some extreme outliers, especially in the cashback column where a few users got way more than everyone else. We might need to deal with these later if they mess up our model.

## 3. Data Preprocessing
Here we will fix missing values, encode the categories, scale numbers, and balance the classes using our files.

In [ ]:
print("Missing values before:", df.isnull().sum().sum())

# 1. Impute missing values
df_clean = impute_missing_values(df, strategy="median")
print("Missing values after impute:", df_clean.isnull().sum().sum())

# 2. Drop CustomerID since it's not useful for machine learning
if "CustomerID" in df_clean.columns:
    df_clean = df_clean.drop(columns=["CustomerID"])

# 3. One Hot Encoding
df_encoded = one_hot_encode_categorical(df_clean)
print("Columns after encoding:", len(df_encoded.columns))

# 4. Min Max Scaling
df_scaled = min_max_scale_numeric(df_encoded)
print("Data scaled!")

# 5. Handle Imbalance
df_final = random_undersample(df_scaled, target_col="Churn")
print("Shape after balancing:", df_final.shape)

## 4. Graphs After Processing
Let's see how the data looks now.

In [ ]:
# Graph 4: Churn is now balanced
plt.figure(figsize=(6,4))
sns.countplot(data=df_final, x="Churn")
plt.title("After Processing: Churn is Balanced!")
plt.savefig("../graphs/preprocessed/05_churn_balanced.png", bbox_inches='tight')
plt.show()

**Insight:** Yay, it worked! After using random undersampling, both classes have the exact same number of rows now. Now our model won't be biased anymore.

In [ ]:
# Graph 5: Distribution of Tenure after scaling
plt.figure(figsize=(6,4))
sns.histplot(df_final["Tenure"], bins=20, kde=True)
plt.title("After Processing: Scaled Tenure Distribution [0, 1]")
plt.savefig("../graphs/preprocessed/06_tenure_scaled.png", bbox_inches='tight')
plt.show()

**Insight:** The shape of the tenure distribution is exactly the same as before, but since we used Min-Max scaling, all the values are squished between 0 and 1 now. This helps the machine learning algorithms work better.

In [ ]:
# Graph 6: Correlation Heatmap of some features
plt.figure(figsize=(8,6))
# I will just pick a few features so the heatmap is not too crazy big
cols_to_plot = ["Churn", "Tenure", "CashbackAmount", "OrderCount", "DaySinceLastOrder"]
# Check if they exist
cols_to_plot = [c for c in cols_to_plot if c in df_final.columns]
sns.heatmap(df_final[cols_to_plot].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("After Processing: Simple Correlation Heatmap")
plt.savefig("../graphs/preprocessed/07_correlation_heatmap.png", bbox_inches='tight')
plt.show()

**Insight:** The correlation heatmap shows how the different columns relate to Churn. We can quickly spot which features might actually be useful for predicting whether a user leaves or not.

## 5. Save the processed data
Finally, we save this ready-to-train dataframe into the processed folder.

In [ ]:
import os
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "cleaned_churn_data.csv")
df_final.to_csv(output_path, index=False)
print("Saved the final preprocessed data to:", output_path)